In [1]:
import os
import json
import uuid
from typing import TypedDict

from google.generativeai.types import GenerationConfig, datetime
from kscLLM.index import ROOT_PATH
from kscLLM.util import get_model, to_markdown, get_current_time
from kscLLM.models import STIXIndicator

from google.generativeai.generative_models import GenerativeModel
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")

set


/home/lukas/Programming/uni/threatintel/gemini/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = get_model()

In [ ]:
with open(ROOT_PATH / "tmp/positives.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Create a STIX Indicator matching the the following SDOs.
A credential access attack is searched. First a token is fetched. This token is then extracted by an SSRF attack which calls the metadata.google.internal service accounts token endpoint.
Match the SDOs by using the message property in each artifact.

{str(observables_bundle)}"""

response_indicator = model.generate_content(
    prompt,
    generation_config=GenerationConfig(response_mime_type="application/json", response_schema=STIXIndicator)
)
response: STIXIndicator = json.loads(response_indicator.text)
response

In [ ]:
indicator = {
    "type": "bundle",
    "id": f"bundle--{uuid.uuid4()}",
    "spec_version": "2.1",
    "objects": [
        response | {
            "type": "indicator",
            "spec_version": "2.1",
            "id": f"indicator--{uuid.uuid4()}",
            "created": get_current_time(),
            "modified": get_current_time(),
            "pattern_type": "stix",
            "pattern_version": "2.1",
            "valid_from": get_current_time()
        }
    ],
}

indicator